# Skyline and Valleyline

Upper and lower envelope extraction for sequential data.

Unlike regression (which fits through the *middle*), these envelopes:
- pass through **actual data points**
- support an **arbitrary number of direction changes**
- exclude only points that genuinely dip inside the boundary

They're rather "musical", all things considered.

In [ ]:
from os import environ

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from amads.polyphony import skyline, envelope

## Plotting helper

In [ ]:
def plot_envelopes(source, title='', show_valley=True, tolerance=0.0):
    """
    Plot raw data, skyline, and (optionally) valleyline for any supported input.
    """
    # Coerce input to points so we can plot the raw data
    raw   = envelope._to_points(source)
    xs    = [p[0] for p in raw]
    ys    = [p[1] for p in raw]

    sky  = envelope.skyline_envelope(source, tolerance=tolerance)
    sky_x, sky_y = zip(*sky) if sky else ([], [])

    fig, ax = plt.subplots(figsize=(12, 4))

    # raw data
    ax.scatter(xs, ys, color='steelblue', s=60, zorder=3, label='data')
    ax.plot(xs, ys, color='steelblue', alpha=0.25, linewidth=1)

    # skyline
    ax.plot(sky_x, sky_y, color='crimson', linewidth=2,
            marker='o', markersize=7, zorder=4, label='skyline')

    if show_valley:
        val  = envelope.valleyline_envelope(source, tolerance=tolerance)
        val_x, val_y = zip(*val) if val else ([], [])
        ax.plot(val_x, val_y, color='seagreen', linewidth=2,
                marker='o', markersize=7, zorder=4, label='valleyline')

    # get current ticks and if spacing < 1, enforce 1 miniumm
    ticks = ax.get_yticks()
    if len(ticks) >= 2 and (ticks[1] - ticks[0] < 1):
        ymin, ymax = ax.get_ylim()
        int_ticks = np.arange(np.floor(ymin), np.ceil(ymax) + 1, 1)
        ax.set_yticks(int_ticks)

    ax.set_title(title or str(source), fontsize=13)
    ax.set_xlabel('onset')
    ax.set_ylabel('pitch')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # print summary
    print(f'  raw data ({len(raw)} pts): {[int(y) for _,y in raw]}')
    print(f'  skyline  ({len(sky)} pts): {[int(y) for _,y in sky]}')
    if show_valley:
        val = envelope.valleyline_envelope(source, tolerance=tolerance)
        print(f'  valley   ({len(val)} pts): {[int(y) for _,y in val]}')

---
## Toy examples

1: '515626738' as a string

In [ ]:
toy_1 = "5156267378"
plot_envelopes(toy_1, title=f'Toy 1: {toy_1}')

2:
`[3, 2, 4, 3, 5, 4, 6, 4, 5, 3, 4, 2, 3]` as a list

In [ ]:
toy_2 = [3, 2, 4, 3, 5, 4, 6, 4, 5, 3, 4, 2, 3]
plot_envelopes(toy_2, title=f'Toy 2: {toy_2}')

3

In [ ]:
toy_3 = '313131313131424231313131313142536'
plot_envelopes(toy_3, title=f'Toy 3: {toy_3}, multiple direction changes')

---
## Input form: plain list of values

In [ ]:
list_input_example = [5,1,5,6,2,6,7,3,8]
plot_envelopes(list_input_example, title=f'List input {list_input_example}')

---
## Effect of `tolerance`

`tolerance=0` (default) keeps every point on the exact boundary.  
A positive tolerance removes additional points whose dip is shallow,
producing a smoother, coarser envelope.

In [ ]:
noisy = [60,59,63,61,65,63,64,62,67,65,66,64,70,68,69]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
tolerances = [0, 1, 2]

for ax, tol in zip(axes, tolerances):
    raw = envelope._to_points(noisy)
    xs  = [p[0] for p in raw]
    ys  = [p[1] for p in raw]
    sky = envelope.skyline_envelope(noisy, tolerance=tol)
    sx, sy = zip(*sky)

    ax.plot(xs, ys, 'o-', color='steelblue', alpha=0.3, label='data')
    ax.plot(sx, sy, 'o-', color='crimson', linewidth=2, label=f'skyline tol={tol}')
    ax.set_title(f'tolerance = {tol}  ({len(sky)} pts retained)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Effect of tolerance on skyline', fontsize=13)
plt.tight_layout()
plt.show()

## <font color='purple'> Exercise: Apply to a Score

Apply this logic to the successive onsets of a score.

- Use your own choice of score, or our suggestion (given by the `url` below)
- Parse the score
    - Hint: `from amads.io.readscore import read_score` works both locally or from an URL.
- Get the pitch and onset pairs.
- Compare yur result against the AMADS implementation (which should be identical).

In [ ]:
# Craig Sapp's encodng of Mozart Piano Sonata No.1
url = "https://raw.githubusercontent.com/craigsapp/mozart-piano-sonatas/refs/heads/main/kern/sonata01-1.krn"

## <font color='crimson'> Solution

In [ ]:
from amads.io.readscore import read_score, set_reader_warning_level
from amads.core.basics import Score, Staff, Note

In [ ]:
set_reader_warning_level("none")

In [ ]:
parsed_score = read_score(url)

In [ ]:
right_hard = parsed_score.find_all(Staff)

In [ ]:
right_hand_pitch_onset_pairs = [(x.onset, x.key_num) for x in parsed_score.list_all(Note)]
right_hand_pitch_onset_pairs

In [ ]:
plot_envelopes(
    right_hand_pitch_onset_pairs[3:29],
    show_valley=False,
    title=f'Mozart 279, Movement 1, Right Hand, Extract'
)

---
## API summary

| Function | Returns | Notes |
|---|---|---|
| `skyline(source, tolerance=0)` | `list[(onset, pitch)]` | upper envelope, exact data points |
| `valleyline(source, tolerance=0)` | `list[(onset, pitch)]` | lower envelope, exact data points |
| `skyline_values(source, tolerance=0)` | `list[float]` | y-values only |
| `valleyline_values(source, tolerance=0)` | `list[float]` | y-values only |

**`source`** accepts:
- `str` of digit characters — e.g. `"313131"`
- `list` of scalars — onsets synthesised as `1, 2, 3, ...`
- `list` of `(onset, pitch)` pairs — arbitrary spacing
- AMADS `Score` object and realted (e.g., `Part`) on which the `.find_all(Note)` operation is valid

End

---